In [ ]:
# ==============================================================================
# DeBERTa-v3-base — FULL RETRAIN FROM SCRATCH: SupCon (Stage 1) + SENTENCE
# head = DIRECT MLP CLASSIFIER (Stage 2, trained on the FROZEN encoder)
# + DOCUMENT head = ATTENTION-POOLING (Stage 3) using the EXACT SAME MLP
# classifier architecture as the sentence head.
!pip install -q -U transformers sentencepiece

import os, re, random, unicodedata, json, copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns
from matplotlib.ticker import MaxNLocator

from torch.utils.data import Dataset, DataLoader, Sampler
from transformers import AutoTokenizer, AutoModel, get_cosine_schedule_with_warmup
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    confusion_matrix, roc_curve, auc, matthews_corrcoef, silhouette_score,
)
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm

from google.colab import drive
drive.mount("/content/drive", force_remount=True)

pd.set_option("display.width", 130)
pd.set_option("display.float_format", lambda x: f"{x:.4f}")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ──────────────────────────────────────────────────────────────────────────
# PUBLICATION STYLE — SAME colors/theme as the rest of the project
# ──────────────────────────────────────────────────────────────────────────
_preferred_fonts = ['Times New Roman', 'Nimbus Roman', 'Liberation Serif', 'DejaVu Serif']
_available = {f.name for f in fm.fontManager.ttflist}
_font_family = next((f for f in _preferred_fonts if f in _available), 'DejaVu Serif')

plt.rcParams.update({
    'font.family': _font_family, 'font.size': 12,
    'axes.titlesize': 14, 'axes.labelsize': 13,
    'xtick.labelsize': 11, 'ytick.labelsize': 11, 'legend.fontsize': 11,
    'figure.dpi': 150, 'savefig.dpi': 300, 'savefig.bbox': 'tight',
    'axes.linewidth': 0.9, 'lines.linewidth': 2.0,
})

IMG_COLORS = {
    "human":    "#1E90FF",
    "ai":       "#FF6347",
    "roc_line": "#2E97F0",
    "val_f1":   "#4CAF50",
    "val_acc":  "#FF9800",
    "attn":     "#8E44AD",
}

def style_ax(ax, ylabel=None, xlabel=None, title=None):
    if title: ax.set_title(title, fontsize=13, fontweight='bold', pad=8)
    if ylabel: ax.set_ylabel(ylabel)
    if xlabel: ax.set_xlabel(xlabel)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_color('#555555')
    ax.spines['bottom'].set_color('#555555')
    ax.grid(axis='y', linestyle='--', linewidth=0.6, alpha=0.35, zorder=0)
    ax.set_axisbelow(True)

def _plot_cm(ax, cm, title, tn, fp, fn, tp):
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax, cbar=False,
               xticklabels=['Human', 'AI'], yticklabels=['Human', 'AI'],
               linewidths=0.5, linecolor='white',
               annot_kws={'fontsize': 12, 'fontweight': 'bold'})
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    ax.set_title(title, fontsize=10, fontweight='bold', pad=6)
    ax.text(0.5, -0.20, f"TN={tn}  FP={fp}  FN={fn}  TP={tp}",
           transform=ax.transAxes, ha='center', fontsize=8, color='#444444')

def render_table_png(df_table, save_path, figsize=None):
    n_rows, n_cols = df_table.shape
    figsize = figsize or (max(6, n_cols * 1.6), 0.5 * n_rows + 1.0)
    fig, ax = plt.subplots(figsize=figsize)
    ax.axis('off')
    tbl = ax.table(cellText=df_table.values, colLabels=df_table.columns,
                   cellLoc='center', loc='center')
    tbl.auto_set_font_size(False); tbl.set_fontsize(10); tbl.scale(1, 1.6)
    for (row, col), cell in tbl.get_celld().items():
        cell.set_edgecolor('#BBBBBB'); cell.set_linewidth(0.6)
        if row == 0:
            cell.set_facecolor('#1D3557'); cell.set_text_props(color='white', weight='bold')
        else:
            cell.set_facecolor('#F5F7FA' if row % 2 == 0 else '#FFFFFF')
    plt.tight_layout()
    fig.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()

# ──────────────────────────────────────────────────────────────────────────
# Sentence tokenizer + SAME 8-step English normalization as the rest of the
# project
# ──────────────────────────────────────────────────────────────────────────
_SENT_SPLIT_RE = re.compile(r'(?<=[.!?])\s+')
MIN_SENT_WORDS = 8

def sent_tokenize_en(text):
    if not isinstance(text, str): return []
    text = re.sub(r'\s*\n+\s*', ' ', text.strip())
    if not text: return []
    return [p.strip() for p in _SENT_SPLIT_RE.split(text) if p.strip()]

_INVISIBLE_CHARS_RE = re.compile('[​‌‍‎‏‪‫‬‭‮﻿]')
_CITATION_RE = re.compile(
    r'\[\d+(?:\s*,\s*\d+)*\]'
    r'|\([^()]{0,60}\d{4}[^()]{0,25}\)'
)
_URL_RE   = re.compile(r'(?:https?://|www\.)\S+')
_EMAIL_RE = re.compile(r'\b[\w.+-]+@[\w-]+\.[\w.-]+\b')
_EMOJI_RE = re.compile(
    "["
    "\U0001F300-\U0001FAFF"
    "\U00002600-\U000027BF"
    "\U0001F1E6-\U0001F1FF"
    "]+", flags=re.UNICODE
)
PUNCTUATION = r";:\"'`“”‘’«»\-–—/\\|@#\$%\^&\*_=~<>\(\)\[\]\{\}"
_NUMERIC_COMMA_SAFE_RE = re.compile(r'(?<!\d),|,(?!\d)')

_CONTRACTIONS = {
    "won't": "will not", "can't": "cannot", "n't": " not",
    "'re": " are", "'s": " is", "'d": " would", "'ll": " will",
    "'ve": " have", "'m": " am",
}
_CONTRACTIONS_RE = re.compile("|".join(re.escape(k) for k in _CONTRACTIONS.keys()))

def normalize_english_text(text):
    if not isinstance(text, str):
        return text
    text = _INVISIBLE_CHARS_RE.sub('', text)
    text = _CITATION_RE.sub(' ', text)
    text = _URL_RE.sub(' ', text)
    text = _EMAIL_RE.sub(' ', text)
    text = _EMOJI_RE.sub(' ', text)
    text = unicodedata.normalize('NFKC', text)
    text = _CONTRACTIONS_RE.sub(lambda m: _CONTRACTIONS[m.group(0)], text)
    text = _NUMERIC_COMMA_SAFE_RE.sub(" ", text)
    text = re.sub(rf"[{PUNCTUATION}]", " ", text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# ==============================================================================
# 1. CONFIG
# ==============================================================================
CFG = {
    "model_name":     "microsoft/deberta-v3-base",
    "max_len":        128,
    "proj_dim":        128,
    "hidden_dim":     512,
    "temperature":    0.07,
    "s1_epochs":      5,
    "s1_lr_enc":      1e-5,   # Stage 1: encoder TRAINABLE at a small LR
    "s1_lr_head":     1e-3,   # Stage 1: proj_head LR
    "s1_warmup":      0.06,
    "s2_epochs":      5,
    "s2_lr_head":     1e-3,   # Stage 2: sentence MLPHead LR (encoder FROZEN this stage)
    "s2_warmup":      0.06,
    "batch_size":     128,
    "weight_decay":   0.01,
    "max_grad_norm":  1.0,
    "th_grid":        np.arange(0.05, 0.96, 0.01),   # sentence-level threshold search
    "domain_col":     "domain",
    "embed_batch":    512,
    "output_dir":     "./deberta_v3_supcon_mlp_sentence_attnpool_doc_results",
    "drive_path":     "/content/drive/MyDrive/english_ai_detector_project/DeBERTaV3_SupCon_MLP_AttnPool",
    # --- Stage 3: Attention-Pooling DOCUMENT head (pooling only — the
    # classifier AFTER pooling is the SAME MLPHead class as the sentence
    # head; see MLPHead / head_dropout below, shared by both heads). ---
    "attn_hidden":       256,
    "head_dropout":      0.3,     # SHARED dropout — used by BOTH the sentence MLPHead and the document MLPHead
    "s3_epochs":         25,
    "s3_patience":       5,
    "s3_lr":             1e-3,
    "s3_weight_decay":   1e-4,
    "s3_batch_size":     32,
    "s3_max_grad_norm":  1.0,
    "attn_th_grid":      np.arange(0.40, 0.61, 0.01),
}
TABLES_DIR = f"{CFG['drive_path']}/tables"
PLOTS_DIR  = f"{CFG['drive_path']}/plots"
os.makedirs(CFG["output_dir"], exist_ok=True)
os.makedirs(TABLES_DIR, exist_ok=True)
os.makedirs(PLOTS_DIR, exist_ok=True)
SUFFIX = "mlp_attnpool_deberta"

print("=" * 70)
print("  DeBERTa-v3 — FULL RETRAIN FROM SCRATCH")
print("  Stage 1: SupCon (encoder trainable) + proj_head")
print("  Stage 2: SENTENCE head = direct MLP classifier, trained on the FROZEN encoder (Cross-Entropy)")
print("  Stage 3: DOCUMENT head = Attention-Pooling (weighted sum) -> SAME MLP classifier architecture as Stage 2")
print("=" * 70)
print(f"  Backbone: {CFG['model_name']}   Device: {DEVICE}")

# ==============================================================================
# 2. DATA LOADING (same fixed splits as the rest of the project)
# ==============================================================================
TRAIN_SPLIT_FILE = "/content/drive/MyDrive/english_ai_detector_project/splits/train_split.csv"
VAL_SPLIT_FILE   = "/content/drive/MyDrive/english_ai_detector_project/splits/val_split.csv"
TEST_SPLIT_FILE  = "/content/drive/MyDrive/english_ai_detector_project/splits/test_split.csv"
DOMAIN_COL = CFG["domain_col"]

def _load_split(path, name):
    d = pd.read_csv(path)
    required = ["text", "label"]
    for extra in [DOMAIN_COL]:
        if extra in d.columns: required.append(extra)
    d = d[required].dropna(subset=["text", "label"]).reset_index(drop=True)
    d["text"] = d["text"].astype(str).apply(normalize_english_text)
    d["label"] = d["label"].astype(int)
    d = d[d["text"].str.len() > 20].reset_index(drop=True)
    if DOMAIN_COL in d.columns:
        d[DOMAIN_COL] = d[DOMAIN_COL].astype(str).str.strip().str.lower()
    print(f"  {name}: {len(d)} documents loaded")
    return d

print("\n" + "=" * 70)
print("  Loading fixed splits")
print("=" * 70)
train_doc_df = _load_split(TRAIN_SPLIT_FILE, "Train")
val_doc_df   = _load_split(VAL_SPLIT_FILE, "Val")
test_doc_df  = _load_split(TEST_SPLIT_FILE, "Test")
HAS_DOMAIN = DOMAIN_COL in train_doc_df.columns
train_doc_lbl = train_doc_df["label"].to_numpy()

def expand_to_sentences(df):
    rows = []
    for doc_idx, row in df.iterrows():
        sents = [s for s in sent_tokenize_en(row["text"]) if len(s.split()) >= MIN_SENT_WORDS]
        if not sents: sents = [row["text"]]
        for s in sents:
            r = {"text": s, "label": row["label"], "doc_idx": doc_idx}
            if HAS_DOMAIN: r[DOMAIN_COL] = row[DOMAIN_COL]
            rows.append(r)
    out = pd.DataFrame(rows)
    out["text"] = out["text"].astype(str); out["label"] = out["label"].astype(int)
    return out.reset_index(drop=True)

print("\n  Expanding to sentence level ...")
train_sent_df = expand_to_sentences(train_doc_df)
train_sent_doc_idx = train_sent_df["doc_idx"].to_numpy()
print(f"  Train sentences: {len(train_sent_df)}")

def build_doc_level(df):
    flat_sents, doc_idx = [], []
    for i, row in df.iterrows():
        sents = [s for s in sent_tokenize_en(row["text"]) if len(s.split()) >= MIN_SENT_WORDS]
        if not sents: sents = [row["text"]]
        for s in sents:
            flat_sents.append(s); doc_idx.append(i)
    return flat_sents, np.array(doc_idx)

val_flat_sents, val_doc_idx = build_doc_level(val_doc_df)
test_flat_sents, test_doc_idx = build_doc_level(test_doc_df)
val_lbl = val_doc_df["label"].to_numpy()
test_lbl = test_doc_df["label"].to_numpy()

def average_sentence_probs_to_docs(sent_probs, doc_idx, n_docs):
    """Mean-of-probabilities aggregation (late fusion) — used ONLY as a
    per-epoch MONITORING signal during Stage 2 (to pick the best epoch by
    validation F1). The final document-level decision comes from Stage 3's
    Attention-Pooling head (early fusion), not from this function."""
    doc_probs = np.full(n_docs, 0.5, dtype=np.float32)
    for i in range(n_docs):
        m = doc_idx == i
        if m.any(): doc_probs[i] = float(sent_probs[m].mean())
    return doc_probs

# ==============================================================================
# 3. DATASET + BALANCED SAMPLER (reused for BOTH Stage 1 and Stage 2)
# ==============================================================================
try:
    tokenizer = AutoTokenizer.from_pretrained(CFG["model_name"])
except Exception as e:
    print(f"  ⚠ Fast tokenizer failed ({e}); falling back to slow tokenizer.")
    tokenizer = AutoTokenizer.from_pretrained(CFG["model_name"], use_fast=False)

class SentDataset(Dataset):
    def __init__(self, d):
        self.texts = d["text"].tolist(); self.labels = d["label"].tolist()
    def __len__(self): return len(self.texts)
    def __getitem__(self, i): return {"text": self.texts[i], "label": self.labels[i]}

def collate_fn(batch):
    enc = tokenizer([b["text"] for b in batch], truncation=True,
                    max_length=CFG["max_len"], padding=True, return_tensors="pt")
    enc["label"] = torch.tensor([b["label"] for b in batch], dtype=torch.long)
    return enc

class BalancedBatchSampler(Sampler):
    def __init__(self, labels, bs):
        self.labels = np.array(labels); self.bs = bs
        self.classes = np.unique(self.labels); self.npc = bs // len(self.classes)
        self.idx = {c: np.where(self.labels == c)[0] for c in self.classes}
    def __iter__(self):
        for c in self.classes: np.random.shuffle(self.idx[c])
        for _ in range(len(self.labels) // self.bs):
            b = []
            for c in self.classes:
                b.extend(np.random.choice(self.idx[c], self.npc, replace=True).tolist())
            random.shuffle(b); yield b
    def __len__(self): return len(self.labels) // self.bs

train_loader = DataLoader(
    SentDataset(train_sent_df),
    batch_sampler=BalancedBatchSampler(train_sent_df["label"].tolist(), CFG["batch_size"]),
    collate_fn=collate_fn, num_workers=2, pin_memory=True,
)

# ==============================================================================
# 4. MODEL — encoder + proj_head (Stage 1) + sentence MLPHead (Stage 2).
# MLPHead is defined ONCE and reused for BOTH the sentence head here and the
# document head's final classifier in Stage 3 below (see chat, 2026-10-22:
# "راس الجملة راس تصنيف مباشر MLP نفس راس المستند").
# ==============================================================================
class ProjectionHead(nn.Module):
    def __init__(self, in_dim, hid, out):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(in_dim, hid), nn.BatchNorm1d(hid),
                                 nn.ReLU(inplace=True), nn.Linear(hid, out))
    def forward(self, x): return F.normalize(self.net(x), dim=-1)

class MLPHead(nn.Module):
    """SHARED classifier architecture (see chat, 2026-10-22) — used TWICE
    in this script: as the SENTENCE head (num_labels=2, Cross-Entropy, on a
    single sentence's mean-pooled embedding) and as the DOCUMENT head's
    final classifier (num_labels=1, BCE-with-logits, on the
    attention-pooled document vector). Literally the same class, same
    hidden-layer shape, same activation, same dropout — only the output
    width differs.
      Linear(in_dim -> in_dim//2) -> ReLU -> Dropout -> Linear(in_dim//2 -> num_labels)
    """
    def __init__(self, in_dim, num_labels=2, dropout_prob=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, in_dim // 2),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout_prob),
            nn.Linear(in_dim // 2, num_labels),
        )
    def forward(self, x):
        return self.net(x)

class SharedModel(nn.Module):
    def __init__(self, name, proj_hid, proj_dim, num_labels=2, cls_dropout=0.3):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(name, torch_dtype=torch.float32)
        hs = self.encoder.config.hidden_size
        self.proj_head = ProjectionHead(hs, proj_hid, proj_dim)
        self.cls_head = MLPHead(hs, num_labels=num_labels, dropout_prob=cls_dropout)   # SENTENCE head
    def get_pooled(self, ids, mask):
        hidden = self.encoder(input_ids=ids, attention_mask=mask).last_hidden_state
        mask_expanded = mask.unsqueeze(-1).float()
        summed = (hidden * mask_expanded).sum(dim=1)
        counts = mask_expanded.sum(dim=1).clamp(min=1e-9)
        return summed / counts
    def forward_contrastive(self, ids, mask):
        return self.proj_head(self.get_pooled(ids, mask))
    def forward_classify(self, ids, mask):
        return self.cls_head(self.get_pooled(ids, mask))

model = SharedModel(CFG["model_name"], CFG["hidden_dim"], CFG["proj_dim"],
                    num_labels=2, cls_dropout=CFG["head_dropout"]).to(DEVICE)
model = model.float()
HIDDEN = model.encoder.config.hidden_size
print(f"\n  Encoder params: {sum(p.numel() for p in model.encoder.parameters())/1e6:.1f}M   Hidden: {HIDDEN}")
print(f"  Sentence MLPHead params: {sum(p.numel() for p in model.cls_head.parameters())/1e3:.1f}K")
print(f"  Encoder dtype: {next(model.encoder.parameters()).dtype}")

AMP_ON = torch.cuda.is_available()

@torch.no_grad()
def encode_sentences(sent_list, batch_size=None, desc="encoding"):
    model.eval()
    bs = batch_size or CFG["embed_batch"]
    embs = []
    for i in tqdm(range(0, len(sent_list), bs), desc=f"  {desc}", leave=False):
        batch = sent_list[i:i + bs]
        enc = tokenizer(batch, truncation=True, max_length=CFG["max_len"],
                        padding=True, return_tensors="pt").to(DEVICE)
        pooled = model.get_pooled(enc["input_ids"], enc["attention_mask"])
        embs.append(pooled.cpu().numpy().astype(np.float32))
    return np.vstack(embs) if embs else np.zeros((0, HIDDEN), dtype=np.float32)

@torch.no_grad()
def predict_probs_classify(sent_list, batch_size=None, desc="predict"):
    """SENTENCE-level prediction — the MLPHead sentence classifier."""
    model.eval()
    bs = batch_size or CFG["embed_batch"]
    probs = []
    for i in tqdm(range(0, len(sent_list), bs), desc=f"  {desc}", leave=False):
        batch = sent_list[i:i + bs]
        enc = tokenizer(batch, truncation=True, max_length=CFG["max_len"],
                        padding=True, return_tensors="pt").to(DEVICE)
        logits = model.forward_classify(enc["input_ids"], enc["attention_mask"])
        p = torch.softmax(logits, dim=-1)[:, 1]
        probs.append(p.cpu().numpy().astype(np.float32))
    return np.concatenate(probs) if probs else np.array([], dtype=np.float32)

# ==============================================================================
# 5. SupCon LOSS
# ==============================================================================
class SupConLoss(nn.Module):
    def __init__(self, t=0.07):
        super().__init__(); self.t = t
    def forward(self, feats, labels):
        dev = feats.device; B = feats.shape[0]
        sim = torch.matmul(feats, feats.T) / self.t
        labels = labels.view(-1, 1)
        pos = (labels == labels.T).float().to(dev)
        diag = torch.eye(B, device=dev); pos = pos - diag
        sim = sim - sim.max(dim=1, keepdim=True)[0].detach()
        exp = torch.exp(sim) * (1 - diag)
        logp = sim - torch.log(exp.sum(dim=1, keepdim=True) + 1e-9)
        npos = pos.sum(dim=1)
        loss = -(pos * logp).sum(dim=1)
        return (loss[npos > 0] / npos[npos > 0]).mean()

supcon_loss = SupConLoss(CFG["temperature"])

def alignment(feats, labels):
    total, count = 0.0, 0
    for c in labels.unique():
        f = feats[(labels == c).nonzero(as_tuple=True)[0]]
        if len(f) < 2: continue
        total += ((f.unsqueeze(0) - f.unsqueeze(1)) ** 2).sum(-1).mean().item(); count += 1
    return total / max(count, 1)

def uniformity(feats):
    return torch.pdist(feats, p=2).pow(2).mul(-2).exp().mean().log().item()

# ==============================================================================
# 6. STAGE 1 — SupCon training, encoder TRAINABLE at a small LR, together
# with proj_head. FULL RETRAIN FROM SCRATCH (starts from the public
# pretrained backbone — no checkpoint of this project is loaded anywhere in
# this script).
# ==============================================================================
print("\n" + "=" * 70)
print("  STAGE 1 — SupCon Training (encoder trainable, small LR + proj_head)")
print("=" * 70)

for p in model.encoder.parameters(): p.requires_grad = True
for p in model.cls_head.parameters(): p.requires_grad = False  # unused this stage
for p in model.proj_head.parameters(): p.requires_grad = True
model.encoder.train()
model.proj_head.train()

opt1 = torch.optim.AdamW([
    {"params": model.encoder.parameters(), "lr": CFG["s1_lr_enc"]},
    {"params": model.proj_head.parameters(), "lr": CFG["s1_lr_head"]},
], weight_decay=CFG["weight_decay"])
n1 = len(train_loader) * CFG["s1_epochs"]
sch1 = get_cosine_schedule_with_warmup(opt1, int(n1 * CFG["s1_warmup"]), n1)
scaler1 = torch.amp.GradScaler('cuda', enabled=AMP_ON)

s1_history = {"loss": [], "align": [], "uniform": []}

for epoch in range(1, CFG["s1_epochs"] + 1):
    model.encoder.train(); model.proj_head.train()
    el, ea, eu = [], [], []
    pbar = tqdm(train_loader, desc=f"  S1 Epoch {epoch:02d}/{CFG['s1_epochs']}", leave=False)
    for batch in pbar:
        ids = batch["input_ids"].to(DEVICE); mask = batch["attention_mask"].to(DEVICE)
        lbls = batch["label"].to(DEVICE)
        with torch.amp.autocast('cuda', enabled=AMP_ON):
            feats = model.forward_contrastive(ids, mask)
            loss = supcon_loss(feats, lbls)
        opt1.zero_grad(); scaler1.scale(loss).backward()
        scaler1.unscale_(opt1)
        nn.utils.clip_grad_norm_(
            list(model.encoder.parameters()) + list(model.proj_head.parameters()),
            CFG["max_grad_norm"])
        scaler1.step(opt1); scaler1.update(); sch1.step()
        with torch.no_grad():
            al = alignment(feats.float(), lbls); un = uniformity(feats.float())
        el.append(loss.item()); ea.append(al); eu.append(un)
        pbar.set_postfix(loss=f"{loss.item():.4f}", align=f"{al:.4f}")
    s1_history["loss"].append(np.mean(el)); s1_history["align"].append(np.mean(ea)); s1_history["uniform"].append(np.mean(eu))
    print(f"  Epoch {epoch:02d}  loss={np.mean(el):.4f}  align={np.mean(ea):.4f}  "
          f"uniform={np.mean(eu):.4f}  lr={sch1.get_last_lr()[0]:.2e}")

print(f"  Stage 1 complete — training ran all {CFG['s1_epochs']} epochs "
      f"(encoder moved at a small LR together with proj_head).")

# ==============================================================================
# 7. STAGE 2 — SENTENCE-level MLPHead classifier, trained on top of the
# NOW-FROZEN encoder (encoder does not move in this stage — only the
# sentence MLPHead is trained). Stage 3 (below) builds on this SAME frozen
# encoder state (the one fixed at the end of Stage 1).
# ==============================================================================
print("\n" + "=" * 70)
print("  STAGE 2 — SENTENCE MLPHead classifier, trained on the FROZEN encoder")
print(f"  (encoder frozen, head lr={CFG['s2_lr_head']:.0e}, "
      f"{CFG['s2_epochs']} epochs, best epoch picked by validation document-level F1 "
      f"@0.5, mean-of-probabilities — monitoring only)")
print("=" * 70)

for p in model.encoder.parameters(): p.requires_grad = False  # frozen for Stage 2 and onward
for p in model.proj_head.parameters(): p.requires_grad = False  # done with Stage 1
for p in model.cls_head.parameters(): p.requires_grad = True
model.encoder.eval()
model.cls_head.train()

ce_loss_fn = nn.CrossEntropyLoss()
opt2 = torch.optim.AdamW([
    {"params": model.cls_head.parameters(), "lr": CFG["s2_lr_head"]},
], weight_decay=CFG["weight_decay"])
n2 = len(train_loader) * CFG["s2_epochs"]
sch2 = get_cosine_schedule_with_warmup(opt2, int(n2 * CFG["s2_warmup"]), n2)
scaler2 = torch.amp.GradScaler('cuda', enabled=AMP_ON)

s2_history = {"loss": [], "acc": [], "val_f1": []}
best_val_f1_s2, best_epoch_s2, best_s2_model_state = -1.0, -1, None

for epoch in range(1, CFG["s2_epochs"] + 1):
    model.encoder.eval(); model.cls_head.train()
    el, preds_all, lbls_all = [], [], []
    pbar = tqdm(train_loader, desc=f"  S2 Epoch {epoch:02d}/{CFG['s2_epochs']}", leave=False)
    for batch in pbar:
        ids = batch["input_ids"].to(DEVICE); mask = batch["attention_mask"].to(DEVICE)
        lbls = batch["label"].to(DEVICE)
        with torch.amp.autocast('cuda', enabled=AMP_ON):
            with torch.no_grad():
                pooled = model.get_pooled(ids, mask)  # encoder frozen — no grad needed
            logits = model.cls_head(pooled)
            loss = ce_loss_fn(logits, lbls)
        opt2.zero_grad(); scaler2.scale(loss).backward()
        scaler2.unscale_(opt2)
        nn.utils.clip_grad_norm_(model.cls_head.parameters(), CFG["max_grad_norm"])
        scaler2.step(opt2); scaler2.update(); sch2.step()
        with torch.no_grad():
            batch_preds = logits.argmax(dim=-1)
        el.append(loss.item())
        preds_all.append(batch_preds.cpu().numpy()); lbls_all.append(lbls.cpu().numpy())
        pbar.set_postfix(loss=f"{loss.item():.4f}")
    preds_all = np.concatenate(preds_all); lbls_all = np.concatenate(lbls_all)
    epoch_acc = float((preds_all == lbls_all).mean())

    epoch_val_sent_probs = predict_probs_classify(val_flat_sents, desc=f"  val check (epoch {epoch:02d})")
    epoch_val_doc_probs = average_sentence_probs_to_docs(epoch_val_sent_probs, val_doc_idx, len(val_doc_df))
    epoch_val_preds = (epoch_val_doc_probs >= 0.5).astype(int)
    epoch_val_f1 = f1_score(val_lbl, epoch_val_preds, zero_division=0)
    model.encoder.eval(); model.cls_head.train()

    s2_history["loss"].append(np.mean(el)); s2_history["acc"].append(epoch_acc)
    s2_history["val_f1"].append(epoch_val_f1)
    is_best = epoch_val_f1 > best_val_f1_s2
    if is_best:
        best_val_f1_s2, best_epoch_s2 = epoch_val_f1, epoch
        best_s2_model_state = copy.deepcopy(model.state_dict())
    print(f"  Epoch {epoch:02d}  loss={np.mean(el):.4f}  train_acc={epoch_acc:.4f}  "
          f"val_f1(doc,th=0.5,mean-pool)={epoch_val_f1:.4f}{'  <- best so far' if is_best else ''}  "
          f"lr_head={sch2.get_last_lr()[0]:.2e}")

print(f"\n  Stage 2 complete — trained all {CFG['s2_epochs']} epochs "
      f"(encoder FROZEN throughout; only the sentence MLPHead trained). "
      f"Best epoch by validation F1: {best_epoch_s2} "
      f"(val F1={best_val_f1_s2:.4f}) — restoring that epoch's FULL model "
      f"(encoder [unchanged] + sentence MLPHead) weights. Stage 3 below "
      f"builds on this SAME frozen encoder.")
model.load_state_dict(best_s2_model_state)
model.eval()

# ==============================================================================
# 7b. STAGE 3 — ATTENTION-POOLING DOCUMENT HEAD, trained on top of the
# SAME frozen encoder used in Stage 2 (unchanged since Stage 1 — the
# encoder never moves after Stage 1 in this script). Each sentence in a
# document gets a learned attention weight (masked softmax over that
# document's own sentences), and the document vector is their WEIGHTED SUM
# of frozen sentence embeddings (early fusion). The document vector is then
# classified by the SAME MLPHead CLASS used as the sentence head in Stage 2
# (see chat, 2026-10-22) — num_labels=1 here (BCE-with-logits) instead of 2
# (Cross-Entropy), same hidden-layer shape/activation/dropout otherwise.
# ==============================================================================
print("\n" + "=" * 70)
print("  STAGE 3 — ATTENTION-POOLING DOCUMENT HEAD (encoder frozen since Stage 1; sentence MLPHead also frozen)")
print(f"  Document classifier reuses the SAME MLPHead architecture as the Stage-2 sentence head")
print(f"  (up to {CFG['s3_epochs']} epochs, early stopping patience={CFG['s3_patience']}, "
      f"best-val-F1 checkpoint kept)")
print("=" * 70)

for p in model.encoder.parameters(): p.requires_grad = False
for p in model.cls_head.parameters(): p.requires_grad = False
model.eval()

print("\n  Extracting frozen sentence embeddings for Stage 3 (train/val/test) ...")
train_sent_emb_s3 = encode_sentences(train_sent_df["text"].tolist(), desc="train sentences (Stage 3)")
val_flat_emb_s3   = encode_sentences(val_flat_sents, desc="val sentences (Stage 3)")
test_flat_emb_s3  = encode_sentences(test_flat_sents, desc="test sentences (Stage 3)")

class DocSeqDataset(Dataset):
    """One item = one document's set of frozen sentence embeddings (variable
    length). Padding/masking is done in the collate fn."""
    def __init__(self, sent_emb, doc_idx, doc_labels, n_docs, emb_dim):
        self.n_docs = n_docs
        self.doc_labels = doc_labels
        self.emb_dim = emb_dim
        self.seqs = [[] for _ in range(n_docs)]
        for i, d in enumerate(doc_idx):
            self.seqs[int(d)].append(sent_emb[i])
    def __len__(self): return self.n_docs
    def __getitem__(self, i):
        seq = self.seqs[i]
        if len(seq) == 0:
            seq = [np.zeros(self.emb_dim, dtype=np.float32)]
        arr = np.stack(seq).astype(np.float32)
        return arr, float(self.doc_labels[i])

def collate_docs(batch):
    arrs, labels = zip(*batch)
    lengths = torch.tensor([a.shape[0] for a in arrs], dtype=torch.long)
    maxT = int(lengths.max().item())
    B = len(arrs); D = arrs[0].shape[1]
    X = torch.zeros(B, maxT, D, dtype=torch.float32)
    mask = torch.zeros(B, maxT, dtype=torch.bool)
    for i, a in enumerate(arrs):
        t = a.shape[0]
        X[i, :t] = torch.from_numpy(a)
        mask[i, :t] = True
    y = torch.tensor(labels, dtype=torch.float32)
    return X, mask, lengths, y

train_doc_seq_ds = DocSeqDataset(train_sent_emb_s3, train_sent_doc_idx, train_doc_lbl, len(train_doc_df), HIDDEN)
val_doc_seq_ds   = DocSeqDataset(val_flat_emb_s3, val_doc_idx, val_lbl, len(val_doc_df), HIDDEN)
test_doc_seq_ds  = DocSeqDataset(test_flat_emb_s3, test_doc_idx, test_lbl, len(test_doc_df), HIDDEN)

train_doc_loader_s3 = DataLoader(train_doc_seq_ds, batch_size=CFG["s3_batch_size"], shuffle=True,
                                 collate_fn=collate_docs, num_workers=0)
val_doc_loader_s3   = DataLoader(val_doc_seq_ds, batch_size=64, shuffle=False, collate_fn=collate_docs)
test_doc_loader_s3  = DataLoader(test_doc_seq_ds, batch_size=64, shuffle=False, collate_fn=collate_docs)

class AttnPoolDocHead(nn.Module):
    """sentence embeddings -> scoring net -> masked softmax -> weighted sum
    (attention pooling) -> MLPHead classifier (SAME class/shape as the
    Stage-2 sentence head, num_labels=1) -> 1 logit. Returns the pooled
    document vector AND the per-sentence attention weights alongside the
    logit (the latter is used by the attention-weight diagnostic below)."""
    def __init__(self, in_dim, attn_hidden=256, dropout=0.3):
        super().__init__()
        self.attn = nn.Sequential(
            nn.Linear(in_dim, attn_hidden), nn.Tanh(), nn.Linear(attn_hidden, 1),
        )
        self.classifier = MLPHead(in_dim, num_labels=1, dropout_prob=dropout)   # SAME architecture as the sentence head
    def forward(self, x, mask):
        scores = self.attn(x).squeeze(-1)
        scores = scores.masked_fill(~mask, float("-inf"))
        weights = torch.softmax(scores, dim=1)
        weights = torch.nan_to_num(weights, nan=0.0)
        pooled = (x * weights.unsqueeze(-1)).sum(dim=1)
        logit = self.classifier(pooled).squeeze(-1)
        return logit, pooled, weights

attn_doc_head = AttnPoolDocHead(HIDDEN, CFG["attn_hidden"], CFG["head_dropout"]).to(DEVICE)
n_attn_params = sum(p.numel() for p in attn_doc_head.parameters())
print(f"  Attention-Pooling document head params: {n_attn_params/1e3:.1f}K  "
      f"(classifier sub-module uses the SAME MLPHead class as the sentence head)")

pos_weight_val = float((train_doc_lbl == 0).sum()) / max(float((train_doc_lbl == 1).sum()), 1.0)
pos_weight = torch.tensor([pos_weight_val], device=DEVICE)
print(f"  BCE pos_weight (neg/pos ratio in train): {pos_weight_val:.4f}")
attn_criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

opt3 = torch.optim.AdamW(attn_doc_head.parameters(), lr=CFG["s3_lr"], weight_decay=CFG["s3_weight_decay"])
sch3 = torch.optim.lr_scheduler.CosineAnnealingLR(opt3, T_max=CFG["s3_epochs"])

@torch.no_grad()
def run_attn_doc_head(loader, return_weights=False):
    attn_doc_head.eval()
    all_probs, all_labels, all_pooled = [], [], []
    all_weights, all_masks = [], []
    for X, mask, lengths, y in loader:
        X = X.to(DEVICE); mask_dev = mask.to(DEVICE)
        logits, pooled, weights = attn_doc_head(X, mask_dev)
        probs = torch.sigmoid(logits).cpu().numpy()
        all_probs.append(probs); all_labels.append(y.numpy()); all_pooled.append(pooled.cpu().numpy())
        if return_weights:
            all_weights.append(weights.cpu().numpy()); all_masks.append(mask.numpy())
    probs_out = np.concatenate(all_probs).astype(np.float32)
    labels_out = np.concatenate(all_labels).astype(int)
    pooled_out = np.concatenate(all_pooled).astype(np.float32)
    if return_weights:
        return probs_out, labels_out, pooled_out, all_weights, all_masks
    return probs_out, labels_out, pooled_out

s3_history = {"train_loss": [], "val_f1": [], "val_acc": [], "val_auc": []}
best_val_f1_s3 = -1.0
best_attn_state = None
epochs_without_improvement = 0

print(f"\n  Training Attention-Pooling document head for up to {CFG['s3_epochs']} epochs "
      f"(best-val-F1 checkpoint kept; EARLY STOPPING after "
      f"{CFG['s3_patience']} epochs with no val-F1 improvement) ...")
for epoch in range(1, CFG["s3_epochs"] + 1):
    attn_doc_head.train()
    batch_losses = []
    pbar = tqdm(train_doc_loader_s3, desc=f"  S3 Epoch {epoch:02d}/{CFG['s3_epochs']}", leave=False)
    for X, mask, lengths, y in pbar:
        X = X.to(DEVICE); mask_dev = mask.to(DEVICE); y = y.to(DEVICE)
        logits, _, _ = attn_doc_head(X, mask_dev)
        loss = attn_criterion(logits, y)
        opt3.zero_grad(); loss.backward()
        nn.utils.clip_grad_norm_(attn_doc_head.parameters(), CFG["s3_max_grad_norm"])
        opt3.step()
        batch_losses.append(loss.item())
        pbar.set_postfix(loss=f"{loss.item():.4f}")
    sch3.step()

    val_probs_epoch, val_labels_epoch, _ = run_attn_doc_head(val_doc_loader_s3)
    val_preds_epoch = (val_probs_epoch >= 0.5).astype(int)
    v_f1 = f1_score(val_labels_epoch, val_preds_epoch, zero_division=0)
    v_acc = accuracy_score(val_labels_epoch, val_preds_epoch)
    v_fpr, v_tpr, _ = roc_curve(val_labels_epoch, val_probs_epoch)
    v_auc = auc(v_fpr, v_tpr)

    s3_history["train_loss"].append(float(np.mean(batch_losses)))
    s3_history["val_f1"].append(float(v_f1))
    s3_history["val_acc"].append(float(v_acc))
    s3_history["val_auc"].append(float(v_auc))

    marker = ""
    if v_f1 > best_val_f1_s3:
        best_val_f1_s3 = v_f1
        best_attn_state = copy.deepcopy(attn_doc_head.state_dict())
        epochs_without_improvement = 0
        marker = "  <- best so far (checkpoint saved)"
    else:
        epochs_without_improvement += 1
    print(f"  Epoch {epoch:02d}  train_loss={np.mean(batch_losses):.4f}  "
          f"val_F1@0.5={v_f1:.4f}  val_Acc@0.5={v_acc:.4f}  val_AUC={v_auc:.4f}{marker}")

    if epochs_without_improvement >= CFG["s3_patience"]:
        print(f"\n  EARLY STOPPING at epoch {epoch} — no val-F1 improvement for "
              f"{CFG['s3_patience']} consecutive epochs (best val F1={best_val_f1_s3:.4f}).")
        break

attn_doc_head.load_state_dict(best_attn_state)
attn_doc_head.eval()
best_epoch_s3 = int(np.argmax(s3_history["val_f1"])) + 1
print(f"\n  Stage 3 complete — restored BEST-VAL-F1 checkpoint "
      f"(epoch {best_epoch_s3}, val F1@0.5={best_val_f1_s3:.4f}).")

attn_val_doc_probs, attn_val_doc_lbl_check, _ = run_attn_doc_head(val_doc_loader_s3)
assert np.array_equal(attn_val_doc_lbl_check, val_lbl), "val label ordering mismatch — check DataLoader shuffle=False"

# ==============================================================================
# 8. VAL/TEST SENTENCE PROBABILITIES (SENTENCE-level, from the Stage-2
# MLPHead). DOCUMENT-level probabilities come from the Stage-3 attention
# head.
# ==============================================================================
print("\n" + "=" * 70)
print("  Scoring val/test sentences with the Stage-2 sentence MLPHead")
print("=" * 70)
val_sent_probs = predict_probs_classify(val_flat_sents, desc="val sentences")
test_sent_probs = predict_probs_classify(test_flat_sents, desc="test sentences")
val_doc_probs = attn_val_doc_probs

# ==============================================================================
# 9. THRESHOLD SEARCH (validation only) — DOCUMENT-level, Stage-3 head.
# Search grid restricted to 0.40-0.60.
# ==============================================================================
best_threshold, best_f1 = 0.5, -1.0
threshold_rows = []
for th in CFG["attn_th_grid"]:
    preds = (val_doc_probs >= th).astype(int)
    f1_th = f1_score(val_lbl, preds, zero_division=0)
    threshold_rows.append({"threshold": round(float(th), 2), "val_f1": round(f1_th, 4)})
    if f1_th > best_f1: best_f1, best_threshold = f1_th, float(th)
threshold_df = pd.DataFrame(threshold_rows)
threshold_df.to_excel(f"{TABLES_DIR}/threshold_search_{SUFFIX}.xlsx", index=False)
print(f"\n  Best threshold on VALIDATION: {best_threshold:.2f}  (val F1={best_f1:.4f})")

fig_th, ax_th = plt.subplots(figsize=(5.5, 4.2))
ax_th.plot(threshold_df["threshold"], threshold_df["val_f1"], color=IMG_COLORS["val_f1"], lw=2)
ax_th.axvline(best_threshold, color="black", ls="--", lw=1, label=f"best={best_threshold:.2f}")
style_ax(ax_th, xlabel="Decision threshold", ylabel="Validation F1 (document-level)",
        title="Threshold Search (Validation) — Attention-Pooling Document Head")
ax_th.legend(fontsize=9)
plt.tight_layout()
fig_th.savefig(f"{PLOTS_DIR}/threshold_search_{SUFFIX}.png"); plt.show()

# ==============================================================================
# 9b. SEPARATE SENTENCE-LEVEL THRESHOLD SEARCH on validation.
# ==============================================================================
val_sent_lbl_flat = val_lbl[val_doc_idx]
best_threshold_sent, best_f1_sent = 0.5, -1.0
threshold_sent_rows = []
for th in CFG["th_grid"]:
    preds = (val_sent_probs >= th).astype(int)
    f1_th = f1_score(val_sent_lbl_flat, preds, zero_division=0)
    threshold_sent_rows.append({"threshold": round(float(th), 2), "val_f1": round(f1_th, 4)})
    if f1_th > best_f1_sent: best_f1_sent, best_threshold_sent = f1_th, float(th)
threshold_sent_df = pd.DataFrame(threshold_sent_rows)
threshold_sent_df.to_excel(f"{TABLES_DIR}/threshold_search_sentence_{SUFFIX}.xlsx", index=False)
print(f"  Best threshold on VALIDATION (sentence-level): {best_threshold_sent:.2f}  (val F1={best_f1_sent:.4f})")

fig_th_sent, ax_th_sent = plt.subplots(figsize=(5.5, 4.2))
ax_th_sent.plot(threshold_sent_df["threshold"], threshold_sent_df["val_f1"], color=IMG_COLORS["val_f1"], lw=2)
ax_th_sent.axvline(best_threshold_sent, color="black", ls="--", lw=1, label=f"best={best_threshold_sent:.2f}")
style_ax(ax_th_sent, xlabel="Decision threshold", ylabel="Validation F1 (sentence-level)",
         title="Threshold Search (Validation, Sentence-level) — Sentence MLPHead")
ax_th_sent.legend(fontsize=9)
plt.tight_layout()
fig_th_sent.savefig(f"{PLOTS_DIR}/threshold_search_sentence_{SUFFIX}.png"); plt.show()

# ==============================================================================
# 10. FINAL TEST EVALUATION — document (Attention-Pooling head) + sentence
# (MLPHead) level
# ==============================================================================
attn_test_doc_probs, attn_test_doc_lbl_check, attn_test_doc_pooled, test_attn_weights, test_attn_masks = \
    run_attn_doc_head(test_doc_loader_s3, return_weights=True)
assert np.array_equal(attn_test_doc_lbl_check, test_lbl), "test label ordering mismatch — check DataLoader shuffle=False"
test_doc_probs = attn_test_doc_probs
test_sent_lbl_flat = test_lbl[test_doc_idx]

test_doc_df = test_doc_df.copy()
test_doc_df["prob_ai"] = test_doc_probs
test_doc_df["pred_label"] = (test_doc_probs >= best_threshold).astype(int)
test_doc_df["true_label"] = test_lbl

accuracy  = accuracy_score(test_lbl, test_doc_df["pred_label"])
f1        = f1_score(test_lbl, test_doc_df["pred_label"])
precision = precision_score(test_lbl, test_doc_df["pred_label"], zero_division=0)
recall    = recall_score(test_lbl, test_doc_df["pred_label"], zero_division=0)
mcc       = matthews_corrcoef(test_lbl, test_doc_df["pred_label"])
fpr, tpr, _ = roc_curve(test_lbl, test_doc_probs)
roc_auc = auc(fpr, tpr)

sent_pred = (test_sent_probs >= best_threshold_sent).astype(int)
sent_accuracy  = accuracy_score(test_sent_lbl_flat, sent_pred)
sent_f1        = f1_score(test_sent_lbl_flat, sent_pred)
sent_precision = precision_score(test_sent_lbl_flat, sent_pred, zero_division=0)
sent_recall    = recall_score(test_sent_lbl_flat, sent_pred, zero_division=0)
sent_mcc       = matthews_corrcoef(test_sent_lbl_flat, sent_pred)
sent_fpr, sent_tpr, _ = roc_curve(test_sent_lbl_flat, test_sent_probs)
sent_roc_auc = auc(sent_fpr, sent_tpr)

print("\n" + "=" * 55)
print("FINAL RESULTS ON TEST SET (Document-level) — ATTENTION-POOLING HEAD")
print(f"  Stage-1 epochs trained : {CFG['s1_epochs']} / {CFG['s1_epochs']}  (encoder trainable, lr={CFG['s1_lr_enc']:.0e})")
print(f"  Stage-2 best epoch     : {best_epoch_s2} / {CFG['s2_epochs']}  (encoder FROZEN, only sentence MLPHead trained)")
print(f"  Stage-3 best epoch     : {best_epoch_s3} / {CFG['s3_epochs']}  (val F1@0.5={best_val_f1_s3:.4f})")
print(f"  Threshold : {best_threshold:.2f}")
print(f"  Accuracy  : {accuracy:.4f}   F1: {f1:.4f}   Precision: {precision:.4f}")
print(f"  Recall    : {recall:.4f}   MCC: {mcc:.4f}   AUC: {roc_auc:.4f}")
print("=" * 55)
print("SENTENCE-LEVEL RESULTS ON TEST SET — MLPHead sentence classifier")
print(f"  Threshold : {best_threshold_sent:.2f}  (SEPARATE sentence-level grid search, val F1={best_f1_sent:.4f})")
print(f"  Accuracy  : {sent_accuracy:.4f}   F1: {sent_f1:.4f}")
print(f"  MCC       : {sent_mcc:.4f}   AUC: {sent_roc_auc:.4f}")
print("=" * 55)

# ==============================================================================
# 11. UNIFIED PERFORMANCE TABLE
# ==============================================================================
test_sent_df_full = pd.DataFrame({
    "label": test_sent_lbl_flat, "prob_ai": test_sent_probs, "pred_label": sent_pred,
})
if HAS_DOMAIN:
    test_sent_df_full[DOMAIN_COL] = test_doc_df[DOMAIN_COL].values[test_doc_idx]

domain_rows = []
if HAS_DOMAIN:
    for dom in sorted(test_doc_df[DOMAIN_COL].unique()):
        sub = test_doc_df[test_doc_df[DOMAIN_COL] == dom]
        dom_fpr, dom_tpr, _ = roc_curve(sub["true_label"], sub["prob_ai"])
        domain_rows.append({
            "Level": f"Document — {dom.capitalize()}", "N": len(sub),
            "Accuracy": round(accuracy_score(sub["true_label"], sub["pred_label"]), 4),
            "F1": round(f1_score(sub["true_label"], sub["pred_label"], zero_division=0), 4),
            "Precision": round(precision_score(sub["true_label"], sub["pred_label"], zero_division=0), 4),
            "Recall": round(recall_score(sub["true_label"], sub["pred_label"], zero_division=0), 4),
            "MCC": round(matthews_corrcoef(sub["true_label"], sub["pred_label"]), 4),
            "AUC": round(auc(dom_fpr, dom_tpr), 4),
        })
domain_perf_df = pd.DataFrame(domain_rows)

overall_doc_row = pd.DataFrame([{
    "Level": "Document — Overall", "N": len(test_doc_df),
    "Accuracy": round(accuracy, 4), "F1": round(f1, 4), "Precision": round(precision, 4),
    "Recall": round(recall, 4), "MCC": round(mcc, 4), "AUC": round(roc_auc, 4),
}])
overall_sent_row = pd.DataFrame([{
    "Level": "Sentence — Overall", "N": len(test_sent_df_full),
    "Accuracy": round(sent_accuracy, 4), "F1": round(sent_f1, 4), "Precision": round(sent_precision, 4),
    "Recall": round(sent_recall, 4), "MCC": round(sent_mcc, 4), "AUC": round(sent_roc_auc, 4),
}])

sent_domain_rows = []
if HAS_DOMAIN:
    for dom in sorted(test_sent_df_full[DOMAIN_COL].unique()):
        sub_s = test_sent_df_full[test_sent_df_full[DOMAIN_COL] == dom]
        s_fpr, s_tpr, _ = roc_curve(sub_s["label"], sub_s["prob_ai"])
        sent_domain_rows.append({
            "Level": f"Sentence — {dom.capitalize()}", "N": len(sub_s),
            "Accuracy": round(accuracy_score(sub_s["label"], sub_s["pred_label"]), 4),
            "F1": round(f1_score(sub_s["label"], sub_s["pred_label"], zero_division=0), 4),
            "Precision": round(precision_score(sub_s["label"], sub_s["pred_label"], zero_division=0), 4),
            "Recall": round(recall_score(sub_s["label"], sub_s["pred_label"], zero_division=0), 4),
            "MCC": round(matthews_corrcoef(sub_s["label"], sub_s["pred_label"]), 4),
            "AUC": round(auc(s_fpr, s_tpr), 4),
        })
sent_domain_df = pd.DataFrame(sent_domain_rows)

parts = [overall_doc_row, domain_perf_df, overall_sent_row]
if not sent_domain_df.empty: parts.append(sent_domain_df)
unified_df = pd.concat(parts, ignore_index=True)
unified_df.to_excel(f"{TABLES_DIR}/unified_performance_table_{SUFFIX}.xlsx", index=False)
render_table_png(unified_df, f"{PLOTS_DIR}/table_unified_performance_{SUFFIX}.png",
                 figsize=(13, 0.55 * len(unified_df) + 1.2))
print(f"\n  Unified table ({len(unified_df)} rows):")
print(unified_df.to_string(index=False))

# ==============================================================================
# 12. CONFUSION MATRICES — Document-Overall, Sentence-Overall, per-domain (both)
# ==============================================================================
domains = sorted(test_doc_df[DOMAIN_COL].unique()) if HAS_DOMAIN else []
n_domain_plots = len(domains)
n_cols = 2 + n_domain_plots + (n_domain_plots if HAS_DOMAIN else 0)
fig_cm, axes_cm = plt.subplots(1, n_cols, figsize=(3.6 * n_cols, 4.2))
if n_cols == 1: axes_cm = [axes_cm]

cm_doc = confusion_matrix(test_lbl, test_doc_df["pred_label"])
tn_d, fp_d, fn_d, tp_d = cm_doc[0,0], cm_doc[0,1], cm_doc[1,0], cm_doc[1,1]
_plot_cm(axes_cm[0], cm_doc, "Document — Overall (Attention-Pooling)", tn_d, fp_d, fn_d, tp_d)

cm_s = confusion_matrix(test_sent_lbl_flat, sent_pred)
tn_s, fp_s, fn_s, tp_s = cm_s[0,0], cm_s[0,1], cm_s[1,0], cm_s[1,1]
_plot_cm(axes_cm[1], cm_s, "Sentence — Overall (MLPHead)", tn_s, fp_s, fn_s, tp_s)

for ax, dom in zip(axes_cm[2:2+n_domain_plots], domains):
    sub = test_doc_df[test_doc_df[DOMAIN_COL] == dom]
    cm_dom = confusion_matrix(sub["true_label"], sub["pred_label"], labels=[0, 1])
    _plot_cm(ax, cm_dom, f"Document — {dom.capitalize()}", *cm_dom.ravel())

for ax, dom in zip(axes_cm[2+n_domain_plots:], domains):
    sub_s = test_sent_df_full[test_sent_df_full[DOMAIN_COL] == dom]
    cm_s_dom = confusion_matrix(sub_s["label"], sub_s["pred_label"], labels=[0, 1])
    _plot_cm(ax, cm_s_dom, f"Sentence — {dom.capitalize()}", *cm_s_dom.ravel())

plt.tight_layout()
fig_cm.savefig(f"{PLOTS_DIR}/confusion_matrices_all_{SUFFIX}.png"); plt.show()
print(f"  Saved -> {PLOTS_DIR}/confusion_matrices_all_{SUFFIX}.png")

# ==============================================================================
# 13. ROC CURVE
# ==============================================================================
fig_roc, ax_roc = plt.subplots(figsize=(5.5, 4.8))
ax_roc.plot(fpr, tpr, color=IMG_COLORS["roc_line"], lw=2.2, label=f"Document, Attn-Pool (AUC={roc_auc:.4f})")
ax_roc.plot(sent_fpr, sent_tpr, color=IMG_COLORS["ai"], lw=2.0, ls="--", label=f"Sentence, MLPHead (AUC={sent_roc_auc:.4f})")
ax_roc.plot([0, 1], [0, 1], linestyle='--', color='#AAAAAA', lw=1)
style_ax(ax_roc, xlabel="False Positive Rate", ylabel="True Positive Rate",
         title="ROC Curve (Test Set)")
ax_roc.legend(loc='lower right', fontsize=9)
plt.tight_layout()
fig_roc.savefig(f"{PLOTS_DIR}/roc_curve_{SUFFIX}.png"); plt.show()

# ==============================================================================
# 14. CONFIDENCE DISTRIBUTION — correct vs incorrect (document-level)
# ==============================================================================
test_doc_df["correct"] = test_doc_df["pred_label"] == test_doc_df["true_label"]
test_doc_df["confidence"] = test_doc_df["prob_ai"].apply(lambda p: p if p >= best_threshold else 1 - p)
fig_conf, ax_conf = plt.subplots(figsize=(5.5, 4.2))
for correct, color, label in [(True, IMG_COLORS["human"], 'Correct'), (False, IMG_COLORS["ai"], 'Incorrect')]:
    subset = test_doc_df.loc[test_doc_df["correct"] == correct, "confidence"]
    if len(subset) > 0:
        ax_conf.hist(subset, bins=25, alpha=0.6, color=color, label=label,
                    density=True, edgecolor='white', linewidth=0.4)
style_ax(ax_conf, xlabel='Model confidence in predicted class', ylabel='Density',
         title='Prediction Confidence: Correct vs Incorrect (Document-level)')
ax_conf.legend(fontsize=9)
plt.tight_layout()
fig_conf.savefig(f"{PLOTS_DIR}/confidence_correct_vs_incorrect_{SUFFIX}.png"); plt.show()

# ==============================================================================
# 15. DOCUMENT-LEVEL t-SNE — uses the Stage-3 attention head's own POOLED
# representation.
# ==============================================================================
sil_score = silhouette_score(attn_test_doc_pooled, test_lbl, metric="cosine",
                             sample_size=min(2000, len(attn_test_doc_pooled)), random_state=SEED)

n_tsne = min(2000, len(attn_test_doc_pooled))
idx_tsne = np.random.default_rng(SEED).choice(len(attn_test_doc_pooled), n_tsne, replace=False)
X_tsne = StandardScaler().fit_transform(attn_test_doc_pooled[idx_tsne])
coords = TSNE(n_components=2, perplexity=40, n_iter=1000, random_state=SEED).fit_transform(X_tsne)

fig_tsne, ax_tsne = plt.subplots(figsize=(6.5, 5.8))
tsne_lbl = test_lbl[idx_tsne]
for lbl_, color, name_ in [(0, IMG_COLORS["human"], "Human"), (1, IMG_COLORS["ai"], "AI")]:
    m = tsne_lbl == lbl_
    ax_tsne.scatter(coords[m, 0], coords[m, 1], c=color, label=name_, alpha=0.6, s=16,
                   edgecolors="white", linewidths=0.2)
style_ax(ax_tsne, xlabel="Dimension 1", ylabel="Dimension 2",
        title=f"t-SNE — Document-level, Attention-Pooled vector (n={n_tsne})")
ax_tsne.legend()
plt.tight_layout()
fig_tsne.savefig(f"{PLOTS_DIR}/tsne_embeddings_document_level_{SUFFIX}.png"); plt.show()
print(f"  Saved -> {PLOTS_DIR}/tsne_embeddings_document_level_{SUFFIX}.png")
print(f"  Silhouette score (attention-pooled document embeddings): {sil_score:.4f}")

# ==============================================================================
# 15b. ATTENTION-WEIGHT DIAGNOSTIC — distribution of each document's MAX
# attention weight, plus a worked example (one correct AI doc + one correct
# Human doc) showing per-sentence attention weights.
# ==============================================================================
flat_doc_weights, flat_max_per_doc = [], []
for w_batch, m_batch in zip(test_attn_weights, test_attn_masks):
    for i in range(w_batch.shape[0]):
        valid = m_batch[i]
        w = w_batch[i][valid]
        flat_doc_weights.append(w)
        flat_max_per_doc.append(float(w.max()) if len(w) > 0 else np.nan)
flat_max_per_doc = np.array(flat_max_per_doc)

fig_attn, (ax_attn1, ax_attn2) = plt.subplots(1, 2, figsize=(12, 4.4))
ax_attn1.hist(flat_max_per_doc[~np.isnan(flat_max_per_doc)], bins=30, color=IMG_COLORS["attn"],
             alpha=0.75, edgecolor='white', linewidth=0.4)
style_ax(ax_attn1, xlabel="Max attention weight within a document", ylabel="Number of documents (test set)",
         title="Attention Concentration Across Test Documents")

test_doc_df_reset = test_doc_df.reset_index(drop=True)
example_doc_idx = {}
for target_label, target_name in [(1, "AI"), (0, "Human")]:
    cand = test_doc_df_reset[(test_doc_df_reset["true_label"] == target_label) & (test_doc_df_reset["correct"])]
    for doc_i in cand.index:
        if len(flat_doc_weights[doc_i]) >= 3:
            example_doc_idx[target_name] = doc_i
            break

bar_width = 0.35
offsets = {"AI": -bar_width / 2, "Human": bar_width / 2}
colors_map = {"AI": IMG_COLORS["ai"], "Human": IMG_COLORS["human"]}
max_sents_shown = max((len(flat_doc_weights[i]) for i in example_doc_idx.values()), default=0)
for name_, doc_i in example_doc_idx.items():
    w = flat_doc_weights[doc_i]
    xs = np.arange(len(w)) + offsets[name_]
    ax_attn2.bar(xs, w, width=bar_width, color=colors_map[name_], alpha=0.85,
                label=f"{name_} doc (correctly classified)")
style_ax(ax_attn2, xlabel="Sentence index within document", ylabel="Attention weight",
         title="Example Documents — Per-Sentence Attention Weights")
if max_sents_shown > 0:
    ax_attn2.set_xticks(range(max_sents_shown))
ax_attn2.legend(fontsize=8)

plt.tight_layout()
fig_attn.savefig(f"{PLOTS_DIR}/attention_weight_diagnostic_{SUFFIX}.png"); plt.show()
print(f"  Saved -> {PLOTS_DIR}/attention_weight_diagnostic_{SUFFIX}.png")
print(f"  Mean of per-document MAX attention weight (test set): {np.nanmean(flat_max_per_doc):.4f}")

# ==============================================================================
# 16. TRAINING CURVES — Stage 1 (SupCon) + Stage 2 (sentence MLPHead) +
# Stage 3 (Attention-Pooling document head)
# ==============================================================================
fig_train, axes_train = plt.subplots(1, 4, figsize=(21, 4.2))
ep1 = range(1, len(s1_history["loss"]) + 1)
axes_train[0].plot(ep1, s1_history["loss"], color=IMG_COLORS["ai"], lw=2.2, marker="o", ms=5, label="SupCon Loss")
axes_train[0].plot(ep1, s1_history["align"], color=IMG_COLORS["human"], lw=1.8, ls="--", marker="s", ms=4, label="Alignment")
style_ax(axes_train[0], xlabel="Epoch", title="Stage 1 — SupCon (encoder trainable)")
axes_train[0].xaxis.set_major_locator(MaxNLocator(integer=True))
axes_train[0].legend(fontsize=9)

ep2 = range(1, len(s2_history["loss"]) + 1)
ax2b = axes_train[1].twinx()
axes_train[1].plot(ep2, s2_history["loss"], color=IMG_COLORS["ai"], lw=2.2, marker="o", ms=5, label="CE Loss")
ax2b.plot(ep2, s2_history["acc"], color=IMG_COLORS["val_acc"], lw=1.8, ls="--", marker="s", ms=4, label="Train Acc")
style_ax(axes_train[1], xlabel="Epoch", ylabel="CE Loss", title="Stage 2 — Sentence MLPHead")
ax2b.set_ylabel("Train Accuracy")
axes_train[1].xaxis.set_major_locator(MaxNLocator(integer=True))
lines1, labels1 = axes_train[1].get_legend_handles_labels()
lines2, labels2 = ax2b.get_legend_handles_labels()
axes_train[1].legend(lines1 + lines2, labels1 + labels2, fontsize=8, loc="center right")

ep3 = range(1, len(s3_history["train_loss"]) + 1)
axes_train[2].plot(ep3, s3_history["train_loss"], color=IMG_COLORS["ai"], lw=2.2, marker="o", ms=5, label="Train BCE Loss")
style_ax(axes_train[2], xlabel="Epoch", title="Stage 3 — Attn-Pool Doc Head Loss")
axes_train[2].xaxis.set_major_locator(MaxNLocator(integer=True))
axes_train[2].legend(fontsize=9)

axes_train[3].plot(ep3, s3_history["val_f1"], color=IMG_COLORS["val_f1"], lw=2.0, marker="o", ms=4, label="Val F1@0.5")
axes_train[3].plot(ep3, s3_history["val_acc"], color=IMG_COLORS["val_acc"], lw=1.8, ls="--", marker="s", ms=4, label="Val Acc@0.5")
axes_train[3].axvline(best_epoch_s3, color="black", ls=":", lw=1.2, label=f"best epoch={best_epoch_s3}")
style_ax(axes_train[3], xlabel="Epoch", title="Stage 3 — Validation F1 / Accuracy")
axes_train[3].xaxis.set_major_locator(MaxNLocator(integer=True))
axes_train[3].legend(fontsize=8)

plt.tight_layout()
fig_train.savefig(f"{PLOTS_DIR}/training_curves_{SUFFIX}.png"); plt.show()

# ==============================================================================
# 17. SUMMARY DASHBOARD
# ==============================================================================
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

ax = axes[0, 0]
sns.heatmap(cm_doc, annot=True, fmt='d', cmap='Blues', ax=ax, cbar=True,
           xticklabels=['Human', 'AI'], yticklabels=['Human', 'AI'],
           linewidths=0.5, linecolor='white', annot_kws={'fontsize': 14, 'fontweight': 'bold'})
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual'); ax.set_title('Confusion Matrix (Attn-Pool)', fontweight='bold')
ax.text(0.5, -0.16, f"TN={tn_d}   FP={fp_d}   FN={fn_d}   TP={tp_d}",
       transform=ax.transAxes, ha='center', fontsize=9, color='#444444')

ax = axes[0, 1]
ax.plot(fpr, tpr, color=IMG_COLORS["roc_line"], lw=2.2, label=f'DeBERTa-v3 SupCon+MLP+AttnPool (AUC={roc_auc:.4f})')
ax.plot([0, 1], [0, 1], color='#999999', lw=1, linestyle='--')
ax.set_xlabel('FPR'); ax.set_ylabel('TPR'); ax.set_title('ROC Curve', fontweight='bold')
ax.legend(loc='lower right', fontsize=9, frameon=True)

ax = axes[0, 2]
human_probs = test_doc_df.loc[test_doc_df["true_label"] == 0, "prob_ai"]
ai_probs    = test_doc_df.loc[test_doc_df["true_label"] == 1, "prob_ai"]
ax.hist(human_probs, bins=40, alpha=0.6, color=IMG_COLORS["human"], label='Human', density=True)
ax.hist(ai_probs, bins=40, alpha=0.6, color=IMG_COLORS["ai"], label='AI', density=True)
ax.axvline(best_threshold, color='black', linestyle='--', lw=1, label=f'threshold={best_threshold:.2f}')
ax.set_xlabel('P(AI)'); ax.set_title('Probability Distribution', fontweight='bold')
ax.legend(fontsize=8)

ax = axes[1, 0]
ax.plot(ep3, s3_history["train_loss"], marker='o', color=IMG_COLORS["ai"], label='Stage-3 BCE Loss (Attn-Pool head)')
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss'); ax.set_title('Stage 3 — Attention-Pooling Head Loss', fontweight='bold')
ax.legend(fontsize=9)

ax = axes[1, 1]
ax.plot(threshold_df["threshold"], threshold_df["val_f1"], color=IMG_COLORS["val_f1"], lw=2, label="Val F1")
ax.axvline(best_threshold, color='black', ls='--', lw=1)
ax.set_xlabel('Threshold'); ax.set_title('Threshold Search (Val)', fontweight='bold')
ax.legend(fontsize=9)

ax = axes[1, 2]
mask_human = tsne_lbl == 0; mask_ai = tsne_lbl == 1
ax.scatter(coords[mask_human, 0], coords[mask_human, 1], s=6, color=IMG_COLORS["human"], alpha=0.6, label='Human')
ax.scatter(coords[mask_ai, 0], coords[mask_ai, 1], s=6, color=IMG_COLORS["ai"], alpha=0.6, label='AI')
ax.set_title(f't-SNE (Document-level, n={n_tsne})', fontweight='bold')
ax.legend(fontsize=8, markerscale=2); ax.set_xticks([]); ax.set_yticks([])

for ax in [axes[0, 1], axes[0, 2], axes[1, 0], axes[1, 1], axes[1, 2]]:
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
    ax.grid(axis='y', linestyle='--', linewidth=0.5, alpha=0.3, zorder=0)
    ax.set_axisbelow(True)

fig.suptitle(f"DeBERTa-v3 — SupCon + MLP sentence head + Attention-Pooling doc head (SAME MLP classifier) "
            f"(s1={CFG['s1_epochs']}ep, s2 best={best_epoch_s2}ep, s3 best={best_epoch_s3}ep) — "
            f"F1={f1:.4f}   AUC={roc_auc:.4f}   Silhouette={sil_score:.4f}",
            fontsize=13, fontweight='bold', y=1.03)
plt.tight_layout()
fig.savefig(f"{PLOTS_DIR}/summary_dashboard_{SUFFIX}.png"); plt.show()
print(f"  Saved -> {PLOTS_DIR}/summary_dashboard_{SUFFIX}.png")

# ==============================================================================
# 18. SAVE
# ==============================================================================
print(f"\n  Saving to Drive -> {CFG['drive_path']}  (suffix: {SUFFIX})")
torch.save({
    "shared_model": model.state_dict(),   # encoder (frozen at its post-Stage-1 state, unchanged through Stage 2/3) + proj_head + sentence MLPHead
    "cfg": CFG,
    "s1_epochs_trained": CFG["s1_epochs"],
    "s2_epochs_trained": CFG["s2_epochs"],
    "s2_best_epoch": best_epoch_s2,
    "s2_best_epoch_val_f1": best_val_f1_s2,
}, os.path.join(CFG["drive_path"], f"encoder_{SUFFIX}.pt"))
torch.save({
    "attn_doc_head_state_dict": attn_doc_head.state_dict(),
    "cfg": CFG,
    "s3_best_epoch": best_epoch_s3,
    "s3_best_val_f1": best_val_f1_s3,
}, os.path.join(CFG["drive_path"], f"attnpool_doc_head_{SUFFIX}.pt"))
with open(os.path.join(CFG["drive_path"], f"config_{SUFFIX}.json"), "w") as f:
    json.dump({
        "th_doc": best_threshold,
        "th_sentence": best_threshold_sent,
        "s1_epochs_trained": CFG["s1_epochs"],
        "s2_epochs_trained": CFG["s2_epochs"],
        "s2_best_epoch": best_epoch_s2,
        "s2_best_epoch_val_f1": round(best_val_f1_s2, 4),
        "s3_best_epoch": best_epoch_s3,
        "s3_best_val_f1": round(best_val_f1_s3, 4),
        "mean_max_attention_weight_test": float(np.nanmean(flat_max_per_doc)),
        "architecture": ("FULL RETRAIN FROM SCRATCH — DeBERTa-v3: Stage 1 "
                         "SupCon with encoder trainable at a small LR "
                         f"({CFG['s1_lr_enc']:.0e}, {CFG['s1_epochs']} "
                         "epochs, together with proj_head) + Stage 2 "
                         "SENTENCE classification using a direct MLPHead "
                         "(Linear->ReLU->Dropout->Linear) trained on the "
                         f"FROZEN encoder (encoder frozen at its post-Stage-1 "
                         f"state, head lr={CFG['s2_lr_head']:.0e}, {CFG['s2_epochs']} "
                         f"epochs, best epoch #{best_epoch_s2} by validation "
                         "F1). Stage 3 keeps the SAME frozen encoder "
                         "(unchanged since Stage 1) and trains an "
                         "ATTENTION-POOLING DOCUMENT head on top of it: "
                         "each sentence gets a learned attention weight "
                         "(masked softmax over the document's own "
                         "sentences), the document vector is their "
                         "WEIGHTED SUM (early fusion), and that pooled "
                         "vector is classified by the SAME MLPHead CLASS "
                         "used by the sentence head in Stage 2 (same "
                         "hidden-layer shape/activation/dropout, only the "
                         "output width differs: 2 logits + Cross-Entropy "
                         "for the sentence head, 1 logit + BCE-with-logits "
                         f"for the document head). Best epoch #{best_epoch_s3} "
                         "selected by validation document-level F1, with "
                         "early stopping."),
    }, f, indent=2)
tokenizer.save_pretrained(os.path.join(CFG["drive_path"], f"tokenizer_{SUFFIX}"))

print("\n" + "=" * 70)
print("  FINAL SUMMARY — SupCon + MLP sentence head + Attention-Pooling doc head (SAME MLP classifier)")
print("=" * 70)
print(f"  Backbone          : {CFG['model_name']}")
print(f"  Stage-1 epochs    : {CFG['s1_epochs']} / {CFG['s1_epochs']}  (encoder trainable, lr={CFG['s1_lr_enc']:.0e})")
print(f"  Stage-2 epochs    : {CFG['s2_epochs']} / {CFG['s2_epochs']}  (encoder FROZEN, only sentence MLPHead trained, lr={CFG['s2_lr_head']:.0e})")
print(f"  Stage-2 best epoch: #{best_epoch_s2}  (val F1={best_val_f1_s2:.4f}, mean-of-probs monitoring)")
print(f"  Stage-3 best epoch: #{best_epoch_s3} / {CFG['s3_epochs']}  (val F1={best_val_f1_s3:.4f}, Attention-Pooling doc head)")
print(f"  Threshold (doc)   : {best_threshold:.2f}  (Attention-Pooling head)")
print(f"  Threshold (sent)  : {best_threshold_sent:.2f}  (sentence MLPHead)")
print(unified_df.to_string(index=False))
print(f"  Silhouette        : {sil_score:.4f}")
print(f"  Drive path        : {CFG['drive_path']}  (files suffixed '_{SUFFIX}')")
print(f"  Tables            : {TABLES_DIR}")
print(f"  Plots             : {PLOTS_DIR}")
print("=" * 70)